# 08 · Health Score, Failure Probability & Maintenance Recommendations

**Phase 16 (health score) + Phase 17 (failure-probability classification) + Phase 18 (prediction uncertainty) of the project plan.**

This closes out the ML pipeline by turning a raw RUL prediction into application-level outputs: a 0-100 health score, binary failure-probability classifiers ("will this engine fail within the next N cycles?"), a rough prediction interval, and a risk category with a suggested action — all clearly derived/decided at the application layer, not NASA-prescribed. This is as far as the plan asked for **before** any API/dashboard/Docker work (explicitly out of scope for now).

Everything here is built on the best regression model saved by `05_tree_models.ipynb`.


In [ ]:
import sys
sys.path.insert(0, "..")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

from src import health as hl

PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../models"
REPORTS_DIR = "../reports"
SUBSET = "FD001"

model = joblib.load(f"{MODELS_DIR}/best_model_{SUBSET}.joblib")
meta = joblib.load(f"{MODELS_DIR}/best_model_{SUBSET}_meta.joblib")
feature_cols = meta["feature_cols"]

train = pd.read_parquet(f"{PROCESSED_DIR}/train_{SUBSET}_features.parquet")
val = pd.read_parquet(f"{PROCESSED_DIR}/val_{SUBSET}_features.parquet")
test = pd.read_parquet(f"{PROCESSED_DIR}/test_{SUBSET}_features.parquet")

X_train, X_val, X_test = train[feature_cols], val[feature_cols], test[feature_cols]
print("Base regression model:", meta["model_name"])


## Phase 16 — Health score

`src/health.health_score` maps predicted RUL to a 0-100 scale: `100 * min(1, RUL_pred / rul_healthy)`. `rul_healthy` (default 125, matching the RUL cap used in `03`/`04`) is the RUL level treated as "fully healthy" — an application choice, not a physical constant.


In [ ]:
val_rul_pred = model.predict(X_val)
val = val.copy()
val["RUL_pred"] = val_rul_pred
val["health_score"] = hl.health_score(val_rul_pred, rul_healthy=125)

example_engine = val["unit_number"].iloc[0]
g = val[val["unit_number"] == example_engine].sort_values("cycle")
plt.plot(g["cycle"], g["health_score"])
plt.xlabel("cycle")
plt.ylabel("health score (100 = healthy, 0 = failed)")
plt.title(f"Engine {example_engine}: predicted health score over time")
plt.show()


## Phase 17 — Failure-probability classification

Binary targets: will the engine fail within the next 10 / 20 / 30 cycles? (`src/health.add_failure_labels`). We train one classifier per horizon on the same feature set as the regressor, then evaluate with metrics suited to an imbalanced, safety-relevant classification problem (precision/recall/F1, ROC-AUC, PR-AUC) rather than accuracy alone.


In [ ]:
THRESHOLDS = (10, 20, 30)
train_labeled = hl.add_failure_labels(train, thresholds=THRESHOLDS, rul_col="RUL")
val_labeled = hl.add_failure_labels(val, thresholds=THRESHOLDS, rul_col="RUL")

for h in THRESHOLDS:
    print(f"fail_within_{h}: {train_labeled[f'fail_within_{h}'].mean():.1%} positive rate (train)")


In [ ]:
classifiers = {}
for h in THRESHOLDS:
    y_train_cls = train_labeled[f"fail_within_{h}"]
    y_val_cls = val_labeled[f"fail_within_{h}"]

    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(X_train, y_train_cls)
    classifiers[h] = clf

    y_val_proba = clf.predict_proba(X_val)[:, 1]
    print(f"--- fail_within_{h} ---")
    print(classification_report(y_val_cls, clf.predict(X_val), zero_division=0))
    print(f"ROC-AUC: {roc_auc_score(y_val_cls, y_val_proba):.3f}  PR-AUC: {average_precision_score(y_val_cls, y_val_proba):.3f}\n")


## Example: failure-probability trajectory for one engine

As an engine approaches failure, `fail_within_20`'s predicted probability should trend upward — a useful sanity check independent of the regression metrics.


In [ ]:
g = val_labeled[val_labeled["unit_number"] == example_engine].sort_values("cycle")
proba_20 = classifiers[20].predict_proba(g[feature_cols])[:, 1]

fig, ax1 = plt.subplots()
ax1.plot(g["cycle"], g["RUL"], color="tab:blue", label="true RUL")
ax1.set_ylabel("true RUL", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(g["cycle"], proba_20, color="tab:red", label="P(fail within 20 cycles)")
ax2.set_ylabel("P(fail within 20 cycles)", color="tab:red")
plt.title(f"Engine {example_engine}: RUL decay vs. predicted failure probability")
plt.show()


## Phase 18 — Prediction uncertainty (quantile interval)

A lightweight uncertainty estimate: two extra gradient-boosting models trained to predict the 10th and 90th percentile of RUL (`loss="quantile"`) instead of the mean, giving a rough `[low, high]` interval around each point prediction rather than a single number with no notion of confidence.


In [ ]:
gbr_low = GradientBoostingRegressor(loss="quantile", alpha=0.1, n_estimators=300, max_depth=3, random_state=42)
gbr_high = GradientBoostingRegressor(loss="quantile", alpha=0.9, n_estimators=300, max_depth=3, random_state=42)
gbr_low.fit(X_train, train["RUL"])
gbr_high.fit(X_train, train["RUL"])

val["RUL_pred_low"] = gbr_low.predict(X_val)
val["RUL_pred_high"] = gbr_high.predict(X_val)

coverage = ((val["RUL"] >= val["RUL_pred_low"]) & (val["RUL"] <= val["RUL_pred_high"])).mean()
print(f"Empirical coverage of the [10th, 90th] percentile interval on validation: {coverage:.1%} (target: 80%)")


In [ ]:
sample = val.sample(n=30, random_state=1).sort_values("RUL")
plt.figure(figsize=(10, 5))
plt.errorbar(
    range(len(sample)),
    sample["RUL_pred"],
    yerr=[sample["RUL_pred"] - sample["RUL_pred_low"], sample["RUL_pred_high"] - sample["RUL_pred"]],
    fmt="o", capsize=3, label="predicted RUL + [10th, 90th] interval",
)
plt.scatter(range(len(sample)), sample["RUL"], color="red", marker="x", label="true RUL", zorder=5)
plt.xlabel("sample (sorted by true RUL)")
plt.ylabel("RUL")
plt.legend()
plt.title("Prediction intervals vs. true RUL (30 random validation rows)")
plt.show()


## Final maintenance table

One row per test engine's most recent observed cycle — the point at which, in a real deployment, you'd be asking "what should we do about this engine right now?" Combines predicted RUL + interval, health score, `fail_within_20` probability, and a LOW/MEDIUM/HIGH risk category with a suggested action (`src/health.RISK_RULES` — application-level cut points at 30/60 cycles, not a NASA standard).


In [ ]:
latest_test = test.sort_values("cycle").groupby("unit_number").tail(1).reset_index(drop=True)
X_latest = latest_test[feature_cols]

latest_test["RUL_pred"] = model.predict(X_latest)
latest_test["RUL_pred_low"] = gbr_low.predict(X_latest)
latest_test["RUL_pred_high"] = gbr_high.predict(X_latest)
latest_test["health_score"] = hl.health_score(latest_test["RUL_pred"], rul_healthy=125)
latest_test["fail_within_20_proba"] = classifiers[20].predict_proba(X_latest)[:, 1]
latest_test["risk"] = hl.risk_category(latest_test["RUL_pred"])
latest_test["recommended_action"] = latest_test["risk"].map(hl.risk_action)

maintenance_table = latest_test[[
    "unit_number", "cycle", "RUL_pred", "RUL_pred_low", "RUL_pred_high",
    "health_score", "fail_within_20_proba", "risk", "recommended_action",
]].sort_values("RUL_pred")

maintenance_table.head(15)


In [ ]:
maintenance_table.to_csv(f"{REPORTS_DIR}/final_maintenance_table_{SUBSET}.csv", index=False)
print(f"saved {REPORTS_DIR}/final_maintenance_table_{SUBSET}.csv ({len(maintenance_table)} engines)")

maintenance_table["risk"].value_counts()


## Summary — and what's intentionally out of scope for now

This notebook completes the ML pipeline requested: data understanding → EDA → leakage-safe feature engineering → baselines → tree models → sequence models → explainability → health/failure-probability/uncertainty outputs, ending in a per-engine maintenance table.

**Deliberately not built yet** (later phases in the plan, once you've reviewed these results):
- Transformer/TCN sequence models, anomaly detection, engine-similarity (DTW) search, sensitivity/what-if analysis.
- FastAPI serving layer (Phase 24), React dashboard (Phase 25), Docker packaging (Phase 28).

Everything above is code only — no cell was executed on your behalf. Run the notebooks in order (`01` → `08`) and use the leaderboards/plots at each step to decide what's worth carrying forward.
